# 01 · Raccolta dati — Andora Real Estate
Tre fonti: **ISTAT IPAB Nord-ovest** (API SDMX reale), **OMI B3 Andora** (baseline di zona), **compravendite** (micro dataset simulato, ancorato all'OMI).

> Correzione metodologica: si usa il livello **Nord-ovest**, non l'indice nazionale, perché una località turistica ligure non segue il trend nazionale.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, numpy as np
pd.set_option('display.width', 120)

## 1. MACRO — Indice ISTAT dei prezzi delle abitazioni, Nord-ovest
Scaricato via SDMX REST (`esploradati.istat.it`), dataflow `143_497`, serie `Q.ITC.59.4.ALL` (trimestrale, Nord-ovest, base 2015=100, abitazioni totali).

In [2]:
from data_fetcher import fetch_istat_ipab
istat = fetch_istat_ipab()   # scarica dal vivo; se offline usa la cache in data/raw
print(istat.shape)
istat.tail(6)

[ISTAT] scaricate 44 osservazioni reali (area=ITC).
(44, 3)


,Data,Trimestre,Indice_Prezzo
38,2024-09-30,2024-Q3,117.2
39,2024-12-31,2024-Q4,117.8
40,2025-03-31,2025-Q1,117.4
41,2025-06-30,2025-Q2,120.9
42,2025-09-30,2025-Q3,122.0
43,2025-12-31,2025-Q4,123.0


## 2. BASELINE — Quotazioni OMI zona B3 (Agenzia delle Entrate)

In [3]:
from data_fetcher import load_omi_b3
omi = load_omi_b3()
omi

,zona,stato,eur_mq_min,eur_mq_max,eur_mq_mid
0,B3,1,2300,2900,2600.0
1,B3,2,2700,3400,3050.0
2,B3,3,3200,4100,3650.0
3,B3,4,3900,5200,4550.0


## 3. MICRO — Compravendite simulate (servizi granulari)
Feature: `metratura, piano, ascensore, classe_energetica, distanza_mare, box_auto`. I prezzi sono ancorati ai valori OMI reali; salvate in `data/processed/`.

In [4]:
from data_fetcher import simulate_micro_dataset
micro = simulate_micro_dataset()
print(micro.shape)
micro.head()

(400, 8)


,prezzo_vendita_eur,metratura,piano,ascensore,classe_energetica,stato_conservazione,distanza_mare_km,box_auto
0,309000,120,4,0,D,2,0.77,1
1,253000,81,6,1,C,2,0.97,0
2,436000,129,6,1,G,3,0.69,1
3,129000,41,6,0,D,3,0.10,0
4,216000,60,7,1,E,2,0.62,1


In [5]:
micro.describe(include='all').T[['count','mean','min','max']]

,count,mean,min,max
prezzo_vendita_eur,400.0,282440.0,82000.0,624000.0
metratura,400.0,85.2125,38.0,129.0
piano,400.0,3.4875,0.0,7.0
ascensore,400.0,0.7225,0.0,1.0
classe_energetica,400,NaN,NaN,NaN
stato_conservazione,400.0,2.49,1.0,4.0
distanza_mare_km,400.0,0.618375,0.06,1.19
box_auto,400.0,0.3875,0.0,1.0
